# Assignment 3: Informed Search

##  Grid Navigation
In this section, you will investigate the problem of navigation on a two dimensional grid with obstacles. The goal is to produce the shortest path between a provided pair of points, taking care to maneuver around the obstacles as needed. Path length is measured in Euclidean distance. Valid directions of movement include up, down, left, right, up-left, up-right, down-left, and down-right. 

### Note:
You are expected to write code where you see **your code here**.  
Make sure you delete the lines with **pass** and **raise NotImplementedError** or your code may not run correctly.

### Task
Your task is to write a function find_path(start, goal, scene) which returns the shortest path from the start point to the goal point that avoids traveling through the obstacles in the grid. 

### Structure/Representation
For this problem, points will be represented as two-element tuples of the form (row, column), and scenes will be represented as two-dimensional lists of Boolean values, with False values corresponding empty spaces and True values corresponding to obstacles. 

### Output
Your output should be the list of points in the path, and should explicitly include both the start point and the goal point. If multiple optimal solutions exist, any of them may be returned. If no optimal solutions exist, or if the start point or goal point lies on an obstacle, you should return the sentinal value None. If start and goal state are the same, return None.

### Implementation
Your implementation should consist of 
* an A* search 
* the straight-line Euclidean distance heuristic. 

Helper functions are allowed and encouraged.

## 1. Euclidean distance
First, let's define a function used for euclidean distance.

In [12]:
############################################################
# Section 1 Grid Navigation - Euclidiean distance
############################################################
def grid_euclidean_distance(current_state, goal_state):
    #pass
    
    # your code here
    #raise NotImplementedError
    from math import sqrt
    return sqrt(sum([(x - y)**2 for (x, y) in zip(current_state, goal_state)]))

In [4]:
##########################
### TEST YOUR SOLUTION ###
##########################

current_state1, goal_state1 = (1,1), (1,1)
assert grid_euclidean_distance(current_state1, goal_state1) == 0

current_state2, goal_state2 = (1,2), (1,1)
assert grid_euclidean_distance(current_state2, goal_state2) == 1

current_state3, goal_state3 = (3,2), (1,2)
assert grid_euclidean_distance(current_state3, goal_state3) == 2

current_state4, goal_state4 = (1,1), (2,2)
assert grid_euclidean_distance(current_state4, goal_state4) == 2**0.5
print("test passed!")

test passed!


## 2. Helper functions
Next, let's define a functions that finds the successors.

In [13]:
# your code here
#raise NotImplementedError

def grid_successors(current, scene):
    # assuming current is not at a false position
    # your code here
    #raise NotImplementedError
    x, y = current
    return tuple([(x + dx, y + dy) for dx in range(-1,2) for dy in range(-1,2) \
                  if x + dx >= 0 and y + dy >= 0 and x + dx < len(scene) and y + dy < len(scene[0]) \
            and not scene[x+dx][y+dy] and not (dx == 0 and dy == 0)])[::-1]

In [25]:
scene1 = [[True, True, True],
         [False, False, True]]
assert grid_successors((1, 2), scene1) == ((1, 1),)
assert grid_successors((0, 1), scene1) == ((1, 1),(1, 0),)
print("test passed!")

test passed!


## 3. Find path
Finally let's implement the path search.

In [14]:
############################################################
# Grid Navigation
############################################################
import collections, itertools, queue, random, copy

def find_path(start, goal, scene):
    #pass

    # your code here
    #raise NotImplementedError
    visited = collections.defaultdict(bool)
    d = collections.defaultdict(int)
    p = {}
    q = queue.PriorityQueue()
    q.put((0, start))
    d[start] = 0
    while q.qsize() > 0:
        _, cur = q.get()
        visited[cur] = True
        if cur == goal:
            break
        for child in grid_successors(cur, scene):
            if not visited[child]:
                p[child] = cur
                d[child] = d[cur] + 1
                q.put((d[child] + grid_euclidean_distance(child, goal), child))
    if not goal in p:
        return None
    path = []
    cur = goal
    while cur != start:
        path = [cur] + path
        cur = p[cur]
    path = [start] + path
    return path

In [47]:
##########################
### TEST YOUR SOLUTION ###
##########################
scene1 = [[False, False, False], 
          [False, True , False], 
          [False, False, False]] 

assert find_path((0, 0), (2, 1), scene1) == [(0, 0), (1, 0), (2, 1)] 

scene2 = [[False, True, False], 
          [False, True, False], 
          [False, True, False]] 
assert find_path((0, 0), (0, 2), scene2) is None
print("test passed!")

test passed!


In [185]:
import collections, itertools, queue, random, copy

def grid_euclidean_distance(current_state, goal_state):
    #raise NotImplementedError
    from math import sqrt
    return sqrt(sum([(x - y)**2 for (x, y) in zip(current_state, goal_state)]))
    
def grid_successors(current, scene):
    # assuming current is not at a false position
    # your code here
    #raise NotImplementedError
    x, y = current
    return tuple([(x + dx, y + dy) for dx, dy in zip([-1,0,0,1], [0,-1,1,0]) \
                  if x + dx >= 0 and y + dy >= 0 and x + dx < len(scene) and y + dy < len(scene[0]) \
                  and scene[x+dx][y+dy]])

def get_priority(algo, g, h):
    if algo == 'A*':
        return g + h
    elif algo == 'Best First':
        return h
    elif algo == 'Uniform Cost':
        return g
    
def find_path(start, goal, scene, algo, h=grid_euclidean_distance):
    # your code here
    #raise NotImplementedError
    visited = collections.defaultdict(bool)
    d = collections.defaultdict(int)
    p = {}
    q = queue.PriorityQueue()
    q.put((0, start))
    d[start] = 0
    while q.qsize() > 0:
        plot_maze(scene, start, goal, algo, visited = list(visited.keys()), fringe = list([x for _, x in q.queue]))
        _, cur = q.get()
        #print(cur)
        visited[cur] = True
        if cur == goal:
            break
        for child in grid_successors(cur, scene):
            if not visited[child]:
                p[child] = cur
                d[child] = d[cur] + 1
                q.put((get_priority(algo, g=d[child], h=h(child, goal)), child))
    if not goal in p:
        return None
    path = []
    cur = goal
    while cur != start:
        path = [cur] + path
        cur = p[cur]
    path = [start] + path
    return path

In [189]:
from skimage.io import imread
from skimage.color import gray2rgb
import numpy as np
import matplotlib.pylab as plt
import matplotlib.patches as mpatches

id = 0

def plot_maze(maze, start, goal, algo, fringe = None, visited = None, path = None):
    global id
    maze = 255*np.array(np.copy(maze))
    #print(maze.shape)
    #print(fringe)
    maze = gray2rgb(maze)
    #print(maze.max())
    maze[start] = [124,252,0]
    maze[goal] = (255,215,0)
    if visited is not None:
        for (x, y) in visited:
            maze[x,y] = 255,192,203 #(127,255,212) #[128, 128, 128]
    if fringe is not None:
        for (x, y) in fringe:
            maze[x,y] = (255,20,147) #[0, 255, 255]
    if path is not None:
        for (x, y) in path:
            maze[x,y] = (255,69,0) #[255, 0, 0]
    #print(maze.shape)

    cmap = {1:[124/255,252/255,0,1],2:[1,215/255,0,1],3:[1,192/255,203/255,1],4:[1,20/255,147/255,1],5:[1,69/255,0,1]}
    labels = {1:'start',2:'goal',3:'visited',4:'fringe',5:'path'}
    ## create patches as legend
    patches =[mpatches.Patch(color=cmap[i],label=labels[i]) for i in cmap]
    
    plt.figure(figsize=(8,8))
    plt.gray(), plt.imshow(maze), plt.axis('off');
    h = 'Euclidean Distance' if algo != 'Uniform Cost' else 'No'
    plt.title(f'{algo} search with {h} heuristic (iter = {id})', size=15)
    plt.legend(handles=patches, loc=4, borderaxespad=0.)
    plt.tight_layout()
    plt.savefig(f'out/out_{id:05d}.png')
    plt.close()
    id += 1
    #plt.show()    

def read_maze():
    maze = imread('maze.jpg', 1)
    maze[maze > 0.5] = 1
    maze[maze <= 0.5] = 0
    maze = maze.astype(int)
    print(maze[13,0], maze.shape, maze[1,28])
    #maze = 1 - maze #~maze
    maze = maze.tolist()
    print(len(maze), len(maze[0]), maze[13][0], maze[1][28])
    return maze

maze = read_maze()

1 (30, 29) 1
30 29 1 1


In [190]:
def run_algo(maze, start, goal, algo):
    global id
    id = 0
    plot_maze(maze, start, goal, algo)
    path = find_path(start, goal, maze, algo)
    plot_maze(maze, start, goal, algo, path=path)

In [193]:
start, goal = (13, 0), (1, 28)

In [ ]:
run_algo(maze, start, goal, 'A*')

In [188]:
run_algo(maze, start, goal, 'Best First')

In [194]:
run_algo(maze, start, goal, 'Uniform Cost')

In [5]:
import imageio
import numpy as np    

#Create reader object for the gif
gif1 = imageio.get_reader('out/out_ucs.gif')
gif2 = imageio.get_reader('out/out_bfs.gif')
gif3 = imageio.get_reader('out/out_astar.gif')

#If they don't have the same number of frame take the shorter
n1, n2, n3 = gif1.get_length(), gif2.get_length(), gif3.get_length()
number_of_frames = max(n1, n2, n3) 
print(n1, n2, n3, number_of_frames)

#Create writer object
#new_gif = imageio.get_writer('output.gif')
new_images = []
shp = (800,800,3)
blnk = 255*np.ones(shp, dtype=np.uint8)
for frame_number in range(number_of_frames):
    if frame_number % 20 == 0: print(frame_number)
    if frame_number < n1-1:
        img1 = gif1.get_next_data()  #else blnk
    if frame_number < n2-1:
        img2 = gif2.get_next_data() #else blnk
    if frame_number < n3-1:
        img3 = gif3.get_next_data() #else blnk
    #here is the magic
    new_image = np.hstack((img1, img2, img3))
    new_images.append(new_image)
    #new_gif.append_data(new_image)
    
for _ in range(20):
    new_images.append(new_image)
    
imageio.mimsave('out/output_search.gif', new_images, duration=10)    
gif1.close()
gif2.close()    
gif3.close()    
#new_gif.close()

310 211 232 310
0
20
40
60
80
100
120
140
160
180
200
220
240
260
280
300
